# RQ1 · Readability over time

**Does the federal web's URL path vocabulary shift from human-readable toward
machine-readable, 2004–2024?** Reported at both token and type level, pooled
across all domains, by segment position, and per domain.

Definitions and caveats: see `00_data_and_methods.ipynb`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root (this notebook lives in analysis/)
sys.path.insert(0, os.path.abspath('.'))
import config, readability as rb, eot_segments as es
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']
CLS_COLORS = {'human':'#2c7fb8','acronym':'#fec44f','machine':'#de2d26'}

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

In [ ]:
seg = es.load_segments(DBS)          # long-form: domain, crawl_year, pos, seg, n, cls
print(f"{len(seg):,} (domain, year, pos, seg) rows across {seg.domain.nunique()} domains")
seg.head()

## 1. Headline — all path positions pooled, human share over time

Falling `human` (or rising `machine`) is the core signal. The token/type gap
tells us whether any change is volume-driven or vocabulary-driven.

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
for lvl, style in [('token','o-'), ('type','s--')]:
    r = es.readability_pct(seg, 'crawl_year', lvl).set_index('crawl_year').reindex(
        [y for y in YEAR_ORDER if y in seg.crawl_year.unique()])
    ax.plot(r.index, r['human'], style, lw=2.5, label=f'human ({lvl})', color='#2c7fb8')
    ax.plot(r.index, r['machine'], style, lw=1.8, label=f'machine ({lvl})', color='#de2d26', alpha=.8)
ax.set_ylim(0,100); ax.set_ylabel('% of path segments'); ax.set_xlabel('crawl year')
ax.set_title('Federal URL path readability over time (all positions, pooled)')
ax.legend(ncol=2); plt.tight_layout(); plt.savefig('rq1_headline.png', dpi=150, bbox_inches='tight'); plt.show()

## 2. Does position matter? (seg1 = top level … seg5 = deep)

Hypothesis: top-level segments stay curated/readable while deeper, generated
levels go machine. Heatmap of **human%** by position × year (type-level).

In [ ]:
by_pos = es.readability_pct(seg, ['pos','crawl_year'], 'type')
piv = by_pos.pivot(index='pos', columns='crawl_year', values='human')
piv = piv[[y for y in YEAR_ORDER if y in piv.columns]]
piv.index = [f'seg{i}' for i in piv.index]
plt.figure(figsize=(9,5))
sns.heatmap(piv, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label':'% human (type-level)'}, linewidths=.5)
plt.title('Human-readability by path position and year'); plt.xlabel('crawl year'); plt.ylabel('')
plt.tight_layout(); plt.savefig('rq1_by_position.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Per-domain — is the trend universal or driven by a few agencies?

Human% (type-level, all positions) by domain × year.

In [ ]:
by_dom = es.readability_pct(seg, ['domain','crawl_year'], 'type')
piv = by_dom.pivot(index='domain', columns='crawl_year', values='human')
piv = piv[[y for y in YEAR_ORDER if y in piv.columns]].reindex(
    [d for d in config.TARGET_DOMAINS if d in piv.index])
plt.figure(figsize=(9,7))
sns.heatmap(piv, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label':'% human (type-level)'}, linewidths=.5)
plt.title('Human-readability by domain and year'); plt.xlabel('crawl year'); plt.ylabel('')
plt.tight_layout(); plt.savefig('rq1_by_domain.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Class composition — where does lost readability go?

Stacked human / acronym / machine, pooled, per year. Acronyms (agency
initialisms) are a distinct middle category — watch whether machine grows at
the expense of human, or acronyms do.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5), sharey=True)
for ax, lvl in zip(axes, ('token','type')):
    r = es.readability_pct(seg, 'crawl_year', lvl).set_index('crawl_year').reindex(
        [y for y in YEAR_ORDER if y in seg.crawl_year.unique()])
    bottom = np.zeros(len(r))
    for c in rb.CLASSES:
        ax.bar(r.index, r[c], bottom=bottom, label=c, color=CLS_COLORS[c])
        bottom += r[c].values
    ax.set_title(f'{lvl}-level'); ax.set_ylabel('% of segments'); ax.set_xlabel('crawl year')
axes[0].legend(); plt.suptitle('Readability class composition over time', y=1.02)
plt.tight_layout(); plt.savefig('rq1_composition.png', dpi=150, bbox_inches='tight'); plt.show()
by_dom.to_csv('rq1_readability_by_domain.csv', index=False)
print('saved rq1_readability_by_domain.csv')